In [5]:
# Report walk-forward robustness scenarios; the primary compact/full choice is frozen in notebook 19 before test evaluation.

In [6]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from project_config import COMPACT_FEATURE_COLUMNS
from pathlib import Path
import inspect
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/modeling_dataset_unprocessed.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
REPORTS = project_paths(ROOT).reports / "07_robustness_visualization"
FIGURES = project_paths(ROOT).figures / "07_robustness_visualization"
CATEGORICAL = ["sector", "industry"]
C_GRID = [0.01, 0.1, 1.0]
MODEL_SETTINGS = {
    "l1_ratio": 0.9,
    "class_weight": "balanced",
    "solver": "saga",
    "max_iter": 10000,
    "tol": 1e-3,
    "random_state": 42,
}
KEYS = ["company_id", "prediction_date"]


def read_events(path):
    frame = pd.read_csv(path, dtype={"company_id": "string"})
    frame["prediction_date"] = pd.to_datetime(
        frame.prediction_date, errors="coerce"
    ).dt.normalize()
    frame["event_key"] = (
        frame.company_id.astype(str)
        + "|"
        + frame.prediction_date.dt.strftime("%Y-%m-%d")
    )
    return frame


def event_key(company, dates):
    date_values = pd.to_datetime(dates, errors="coerce").dt.normalize()
    return company.astype(str) + "|" + date_values.dt.strftime("%Y-%m-%d")


events = read_events(PROCESSED / "modeling_dataset_unprocessed.csv")
if events.event_key.duplicated().any():
    raise ValueError("Modeling data has duplicate company-event rows")
events = events.set_index("event_key", drop=False)
events["bankruptcy_date"] = pd.to_datetime(
    events.bankruptcy_date, errors="coerce"
).dt.normalize()
events["metadata_outcome_end_date"] = pd.to_datetime(
    events.metadata_outcome_end_date, errors="coerce"
).dt.normalize()
walk_path = PROCESSED / "walk_forward_fold_ids.csv"
fold_manifest = pd.read_csv(
    walk_path,
    usecols=[
        "fold",
        "split",
        "company_id",
        "prediction_date",
        "metadata_event_id",
        "company_group_id",
        "metadata_outcome_end_date",
    ],
    dtype={
        "company_id": "string",
        "metadata_event_id": "string",
        "company_group_id": "string",
    },
)
fold_manifest = fold_manifest.loc[fold_manifest.fold.ne("final_test")].copy()
fold_manifest["prediction_date"] = pd.to_datetime(
    fold_manifest.prediction_date, errors="raise"
).dt.normalize()
fold_manifest["metadata_outcome_end_date"] = pd.to_datetime(
    fold_manifest.metadata_outcome_end_date, errors="raise"
).dt.normalize()
fold_manifest["event_key"] = (
    fold_manifest.company_id.astype(str)
    + "|"
    + fold_manifest.prediction_date.dt.strftime("%Y-%m-%d")
)
FOLDS = sorted(fold_manifest.fold.unique())

original = [c for c in events if c.startswith("original_")]
derived = [c for c in events if c.startswith("derived_")]
trends = [c for c in events if c.startswith("trend_")]
market = [c for c in events if c.startswith("market_")]
macro = [c for c in events if c.startswith("macro_")]
missing = [c for c in events if c.startswith("missing_")]
market_sources = {
    "adjusted_price",
    "market_cap",
    "shares_outstanding",
    "volume",
    "turnover",
}
market_original = [
    c
    for c in original
    if c.removeprefix("original_") in market_sources
    or c.startswith("original_market_monthly_price_")
]
financial_original = [c for c in original if c not in market_original]
market_derived = [c for c in derived if c == "derived_market_to_book"]
financial_derived = [c for c in derived if c not in market_derived]
market_trends = [
    c
    for c in trends
    if any(c.startswith(f"trend_{name}_") for name in market_sources)
    or "market_history" in c
]
financial_trends = [c for c in trends if c not in market_trends]
financial = financial_original + financial_derived + financial_trends
market_features = market_original + market_derived + market_trends + market
full_features = original + derived + trends + market + macro
compact_features = [c for c in COMPACT_FEATURE_COLUMNS if c in events.columns]
if not compact_features:
    raise ValueError("No configured compact features appear in the event data")
feature_sets = {
    "all_features": full_features,
    "full_features": full_features,
    "compact_features": compact_features,
    "financial_only": financial,
    "financial_plus_market": financial + market_features,
    "financial_plus_macro": financial + macro,
    "original_levels_only": original,
    "levels_only": original,
    "derived_ratios_only": derived,
    "trends_only": trends,
    "trends_1y": [c for c in trends if "_1y" in c or "has_1y" in c],
    "trends_2y": [c for c in trends if "_2y" in c or "has_2y" in c],
    "trends_3y": [c for c in trends if "_3y" in c or "has_3y" in c],
    "trends_1to3y": trends,
}

C:\Users\MyPC\AppData\Local\Temp\ipykernel_21224\1459965115.py:57: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  frame["event_key"] = (


In [7]:
# Define matched cohorts, training-fitted preprocessing, and retained sensitivity scenarios.
link_cache = {}


def matched_cohort(ratio, replacement, split_map):
    if ratio not in link_cache:
        links = pd.read_csv(
            PROCESSED / f"matched_event_sample_1to{ratio}.csv",
            dtype={"bankrupt_company_id": "string", "control_company_id": "string"},
        )
        links["bank_key"] = event_key(
            links.bankrupt_company_id, links.bankrupt_prediction_date
        )
        links["control_key"] = event_key(
            links.control_company_id, links.control_pseudo_event_date
        )
        link_cache[ratio] = links
    links = link_cache[ratio].copy()
    links["bank_split"] = links.bank_key.map(split_map)
    links["control_split"] = links.control_key.map(split_map)
    links = links.loc[
        links.bank_split.notna() & links.bank_split.eq(links.control_split)
    ].copy()
    if not replacement:
        links = links.sort_values(
            ["match_rank", "size_distance", "bankrupt_prediction_date"], kind="stable"
        )
        links = links.drop_duplicates("control_company_id", keep="first")
    case_keys = links.bank_key.drop_duplicates()
    control_counts = links.control_key.value_counts()
    cases = events.reindex(case_keys).dropna(subset=["company_id"]).copy()
    controls = events.reindex(control_counts.index).dropna(subset=["company_id"]).copy()
    cases["sampling_weight"] = 1.0
    controls["sampling_weight"] = control_counts.reindex(controls.index).astype(float)
    cohort = pd.concat([cases, controls], axis=0).copy()
    cohort["split"] = pd.concat(
        [
            links.drop_duplicates("bank_key").set_index("bank_key").bank_split,
            links.drop_duplicates("control_key").set_index("control_key").control_split,
        ]
    ).reindex(cohort.index)
    cohort["matched_control_links"] = len(links)
    cohort["controls_per_case"] = len(links) / max(len(case_keys), 1)
    return cohort


def scenario_features(name, include_missing):
    selected = list(dict.fromkeys(feature_sets[name]))
    if include_missing:
        bases = {
            c.removeprefix("original_") for c in selected if c.startswith("original_")
        }
        bases.update(
            c.removeprefix("trend_")
            .split("_change_")[0]
            .split("_growth_")[0]
            .split("_slope_")[0]
            for c in selected
            if c.startswith("trend_")
        )
        selected.extend(c for c in missing if c.removeprefix("missing_") in bases)
    return list(dict.fromkeys(c for c in selected if c in events.columns))


def make_model(numeric_columns, C_value):
    categorical = [c for c in CATEGORICAL if c in events.columns]
    numeric = Pipeline(
        [
            ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scale", StandardScaler()),
        ]
    )
    category = Pipeline(
        [
            (
                "impute",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )
    prep = ColumnTransformer(
        [("numeric", numeric, numeric_columns), ("categorical", category, categorical)],
        remainder="drop",
    )
    params = dict(MODEL_SETTINGS, C=C_value)
    if (
        inspect.signature(LogisticRegression).parameters["penalty"].default
        != "deprecated"
    ):
        params["penalty"] = "elasticnet"
    return Pipeline(
        [("preprocess", prep), ("classifier", LogisticRegression(**params))]
    )


def score(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    tn, fp, fn, tp = confusion_matrix(
        labels, (probabilities >= 0.5).astype(int), labels=[0, 1]
    ).ravel()
    return {
        "roc_auc": roc_auc_score(labels, probabilities)
        if np.unique(labels).size == 2
        else np.nan,
        "pr_auc_average_precision": average_precision_score(labels, probabilities),
        "balanced_accuracy": balanced_accuracy_score(
            labels, (probabilities >= 0.5).astype(int)
        ),
        "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "f1": f1_score(labels, (probabilities >= 0.5).astype(int), zero_division=0),
        "brier_score": brier_score_loss(labels, probabilities),
    }


scenarios = []


def add_scenario(
    name,
    comparison,
    feature_set="all_features",
    ratio=2,
    replacement=True,
    horizon=12,
    include_missing=True,
    primary=False,
):
    scenarios.append(
        {
            "scenario": name,
            "comparison": comparison,
            "feature_set": feature_set,
            "matching_ratio": ratio,
            "control_replacement": replacement,
            "horizon_months": horizon,
            "missing_indicators": include_missing,
            "primary_candidate": primary,
        }
    )


# Keep existing feature, missingness, matching, and outcome-horizon scenarios.
for name in [
    "financial_only",
    "financial_plus_market",
    "financial_plus_macro",
    "original_levels_only",
    "derived_ratios_only",
    "trends_only",
    "trends_1y",
    "trends_2y",
    "trends_3y",
    "trends_1to3y",
]:
    add_scenario(name, "feature_family", feature_set=name)
add_scenario(
    "compact_specification",
    "feature_specification",
    "compact_features",
    include_missing=False,
)
add_scenario(
    "full_specification", "feature_specification", "all_features", include_missing=True
)
add_scenario(
    "all_features_with_missing_indicators", "missingness", include_missing=True
)
add_scenario(
    "all_features_without_missing_indicators", "missingness", include_missing=False
)
for ratio in [1, 2, 3]:
    add_scenario(
        f"matching_1to{ratio}", "matching_ratio", ratio=ratio, replacement=True
    )
    add_scenario(
        f"matching_1to{ratio}_with_replacement",
        "control_replacement",
        ratio=ratio,
        replacement=True,
    )
    add_scenario(
        f"matching_1to{ratio}_without_replacement",
        "control_replacement",
        ratio=ratio,
        replacement=False,
    )
# Add explicit feature-specification candidates with matching and missingness held fixed.
add_scenario(
    "compact_features",
    "primary_feature_specification",
    "compact_features",
    include_missing=False,
    primary=True,
)
add_scenario(
    "full_features",
    "primary_feature_specification",
    "full_features",
    include_missing=False,
    primary=True,
)
add_scenario("levels_only", "feature_family", "levels_only")
for horizon in [6, 12]:
    add_scenario(f"outcome_{horizon}m", "outcome_horizon", horizon=horizon)

In [8]:
# Tune each scenario's regularization using walk-forward folds and save pooled OOF predictions.
def eligible_cohort(cohort, horizon):
    end = cohort.prediction_date + pd.DateOffset(months=horizon)
    complete = (
        cohort.metadata_outcome_end_date.notna()
        & cohort.metadata_outcome_end_date.ge(end)
    )
    keep = cohort.bankruptcy_date.notna() | complete
    cohort = cohort.loc[keep].copy()
    end = cohort.prediction_date + pd.DateOffset(months=horizon)
    cohort["target"] = (
        cohort.bankruptcy_date.notna() & cohort.bankruptcy_date.le(end)
    ).astype(int)
    return cohort


fold_rows = []
selected_oof = {}
scenario_choices = []
fold_split_maps = {}
for fold in FOLDS:
    part = fold_manifest.loc[fold_manifest.fold.eq(fold)]
    fold_split_maps[fold] = part.set_index("event_key").split

for setting in scenarios:
    predictors = scenario_features(
        setting["feature_set"], setting["missing_indicators"]
    )
    if not predictors:
        raise ValueError(f"No predictors for {setting['scenario']}")
    candidate_outputs = {C_value: [] for C_value in C_GRID}
    for fold in FOLDS:
        cohort = eligible_cohort(
            matched_cohort(
                setting["matching_ratio"],
                setting["control_replacement"],
                fold_split_maps[fold],
            ),
            setting["horizon_months"],
        )
        train = cohort.loc[cohort.split.eq("train")]
        validation = cohort.loc[cohort.split.eq("validation")]
        categorical = [c for c in CATEGORICAL if c in cohort.columns]
        if (
            train.empty
            or validation.empty
            or train.target.nunique() < 2
            or validation.target.nunique() < 2
        ):
            raise ValueError(
                f"{setting['scenario']}/{fold}: insufficient matching or target classes"
            )
        for C_value in C_GRID:
            model = make_model(predictors, C_value)
            model.fit(
                train[predictors + categorical],
                train.target,
                classifier__sample_weight=train.sampling_weight.to_numpy(dtype=float),
            )
            probability = model.predict_proba(validation[predictors + categorical])[
                :, 1
            ]
            fold_metrics = score(validation.target, probability)
            fold_rows.append(
                {
                    **setting,
                    "fold": fold,
                    "C": C_value,
                    "train_rows": len(train),
                    "validation_rows": len(validation),
                    "validation_bankruptcies": int(validation.target.sum()),
                    **fold_metrics,
                }
            )
            prediction = validation[KEYS + ["metadata_event_id", "target"]].copy()
            prediction["fold"] = fold
            prediction["probability"] = probability
            candidate_outputs[C_value].append(prediction)
    scenario_fold = pd.DataFrame(
        [r for r in fold_rows if r["scenario"] == setting["scenario"]]
    )
    candidate_summary = scenario_fold.groupby("C", as_index=False).agg(
        mean_fold_pr_auc=("pr_auc_average_precision", "mean"),
        mean_fold_roc_auc=("roc_auc", "mean"),
        mean_fold_brier=("brier_score", "mean"),
    )
    chosen = candidate_summary.sort_values(
        ["mean_fold_pr_auc", "mean_fold_roc_auc", "mean_fold_brier", "C"],
        ascending=[False, False, True, True],
        kind="stable",
    ).iloc[0]
    oof = pd.concat(candidate_outputs[float(chosen.C)], ignore_index=True)
    if oof.metadata_event_id.duplicated().any():
        raise ValueError(f"{setting['scenario']}: duplicated OOF event")
    pooled = score(oof.target, oof.probability)
    summary = {
        **setting,
        "split": "oof_validation",
        "events": len(oof),
        "companies": oof.company_id.nunique(),
        "bankrupt_events": int(oof.target.sum()),
        "prevalence": float(oof.target.mean()),
        "predictor_count": len(predictors),
        "selected_C": float(chosen.C),
        "mean_fold_validation_pr_auc": float(chosen.mean_fold_pr_auc),
        "pooled_oof_pr_auc": pooled["pr_auc_average_precision"],
        **pooled,
    }
    selected_oof[setting["scenario"]] = oof.assign(
        scenario=setting["scenario"], selected_C=float(chosen.C)
    )
    scenario_choices.append(summary)

fold_validation_results = pd.DataFrame(fold_rows)
oof_results = pd.DataFrame(scenario_choices)
# The primary choice was frozen in notebook 19 using OOF PR-AUC before final-model fitting.
selection_path = MODELS / "primary_feature_specification.json"
primary_specification_record = json.loads(selection_path.read_text(encoding="utf-8"))
primary_scenario = f"{primary_specification_record['primary_specification']}_features"
primary_candidates = oof_results.loc[oof_results.primary_candidate].copy()
primary_choice = oof_results.loc[oof_results.scenario.eq(primary_scenario)].iloc[0]


NameError: name 'MODELS' is not defined

In [ ]:
# Save walk-forward robustness results without reopening the final test set.
final_test_results = pd.DataFrame()
robustness_results = oof_results.copy()

# Run the stage analysis after saving walk-forward sensitivity results.
import runpy

analysis = runpy.run_path(ROOT / "analysis" / "07_robustness_visualization.py")
analysis["main"](globals())
